# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 4: Solving Tasks with Prompting LLMs</font>

# <font color="#003660">Notebook 1: Leveraging the **Large** (or Small) Language Model Infrastructure</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... you know the basics of large language models and prompting. <br>
        ... you know how to use OpenAI API for prompting LLMs, <br>
        ... how to use its generate and chat api.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

These documentations can help you for your projects:
* OpenAI API: [OpenAI Python SDK](https://platform.openai.com/docs/api-reference/responses/create?lang=python)
* Ollama provides OpenAI API compatible endpoints: [Ollama](https://github.com/ollama/ollama), [Ollama Python](https://github.com/ollama/ollama-python), [Ollama OpenAI Compatibility](https://docs.ollama.com/api/openai-compatibility)
* HuggingFace provides OpenAI API compatible endpoints: [HuggingFace Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [HuggingFace Endpoints](https://huggingface.co/docs/inference-endpoints/index)

## From Language Model to Large Language Model

Last session you learned everything about decoder-only language models.

This session we will shift from (decoder-only) language models to *large* language models (LLMs).

The are called *large* language models, simply because the are *large*.

The are trained on vast amounts of textual data and also contain billions of parameters ([Shanahan et al., 2023](https://doi.org/10.1038/s41586-023-06647-8)).

## Why aren't we using HuggingFace?
In HuggingFace you can simply use a [HuggingFace Pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines)

Usually we would also do this in class. 

However, in deployment scenarios with many users one would not use HuggingFace Pipelines but some endpoint that is in the cloud.

As we have a scenario with many users (you all), we will work with models *deployed* to dedicated endpoints for multi users.

## OpenAI isn't this proprietary?

The answer is simple: Yes OpenAIs models are proprietary. 

Therefore, we won't use them. *But* diverse providers (some examples below) are compatible with the OpenAI API:

* Ollama allows to local host OpenAI API compatible endpoints, running LLMs (usually SLMs) on-premises: [Ollama](https://github.com/ollama/ollama), [Ollama Python](https://github.com/ollama/ollama-python), [Ollama OpenAI Compatibility](https://docs.ollama.com/api/openai-compatibility)
* HuggingFace provides OpenAI API compatible endpoints: [HuggingFace Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [HuggingFace Inference Endpoints](https://huggingface.co/docs/inference-endpoints/index)
* Many other providers host OpenAI API compatible endpoints.

## Setup

You need to install the OpenAI Endpoint.
```
pip install openai
```
Just uncomment the code below.

In [ ]:
# This command creates a new conda environment called session_06 including Python 3.11, JupyterLab, and ipywidgets.
!pip install openai

In [ ]:
BASE_URL = "http://65.108.32.135:11434/v1"
API_KEY = "ollama"

## OpenAI-Compatible APIs

### Setting up a Client

Today we will use the python implementation of [OpenAI](https://platform.openai.com/docs/api-reference/responses/create?lang=python) hosted on a dedicated server with [Ollama](https://ollama.com/).

First, we need to define a `client`.

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    # here the url for the custom endpoint and the API key for authentication
	base_url=BASE_URL,  # example for ollama server
    # here the API key for authentication
	api_key=API_KEY,
)

This client allows you to call all models provided via the endpoint.

The code below lists all available free models.

In [ ]:
models = client.models.list()
for model in models:
    print(model.id)

## Our first LLM-generated text

But first we will work with Llama 3.2 ([MetaAI, 2024](https://ai.meta.com/blog/llama-3-2-connect-2024-vision-edge-mobile-devices/)).

I will explain the parts of the OpenAI Python SDK in the session, if you were not able to attend, refer to the [OpenAI Python SDK](https://platform.openai.com/docs/api-reference/responses/create?lang=python).

In [ ]:
prompt = "We will, we will rock you\nWe will, we will rock you\n"

response = client.chat.completions.create(
    extra_body={},
    temperature=0,
    seed=42,
    max_tokens=512,
    model="llama3.2:1b-text-q4_K_M",
    messages=[
        # this is how messages look like in the OpenAI API
        {
            "role": "user", # you always have a role: user (your prompt), assistant (the model's response), system (system prompt for instructions)
            "content": prompt
        }
    ]
)
print("\033[93mJSON response:\033[0m")
print(response.model_dump_json(indent=4))
print()
print("\033[93mOur input prompt:\033[0m")
print(prompt)
print("\033[93mGenerated text:\033[0m")
if hasattr(response.choices[0].message, 'reasoning'):
    print("<think>")
    print(response.choices[0].message.reasoning)
    print("</think>")

print(response.choices[0].message.content)

Looking at the orginal lyrics (below, taken from [Genius](https://genius.com/Queen-we-will-rock-you-lyrics)), this looks wrong.

In [ ]:
"""[Verse 1]
Buddy, you're a boy, make a big noise
Playing in the street
Gonna be a big man someday
You got mud on your face
You big disgrace
Kickin' your can all over the place, singin'

[Chorus]
We will, we will rock you
We will, we will rock you

[Verse 2]
Buddy, you're a young man, hard man
Shouting in the street
Gonna take on the world someday
You got blood on your face
You big disgrace
Waving your banner all over the place

[Chorus]
We will, we will rock you (Sing it, oh)
We will, we will rock you

[Verse 3]
Buddy, you're an old man, poor man
Pleading with your eyes
Gonna make you some peace someday
You got mud on your face
Big disgrace
Somebody better put you back into your place
See upcoming rock shows
Get tickets for your favorite artists
You might also like
The Tortured Poets Department
Taylor Swift
So Long, London
Taylor Swift
THE HEART PART 6
Drake
[Chorus]
We will, we will rock you, sing it
We will, we will rock you
Everybody, we will, we will rock you
Hmm, we will, we will rock you
Alright

[Guitar Solo]"""

Let's try give it some more context:

In [ ]:
prompt = """[Verse 1]
Buddy, you're a boy, make a big noise
Playing in the street
Gonna be a big man someday
You got mud on your face
You big disgrace
Kickin' your can all over the place, singin'

[Chorus]
We will, we will rock you
We will, we will rock you

[Verse 2]
Buddy, you're a young man, hard man
Shouting in the street
Gonna take on the world someday
You got blood on your face
You big disgrace
Waving your banner all over the place

[Chorus]
We will, we will rock you (Sing it, oh)
We will, we will rock you

[Verse 3]
"""

response = client.chat.completions.create(
    extra_body={},
    temperature=0,
    seed=42,
    max_tokens=512,
    model="llama3.2:1b-text-q4_K_M",
    messages=[
        # this is how messages look like in the OpenAI API
        {
            "role": "user", # you always have a role: user (your prompt), assistant (the model's response), system (system prompt for instructions)
            "content": prompt
        }
    ]
)
print("\033[93mJSON response:\033[0m")
print(response.model_dump_json(indent=4))
print()
print("\033[93mOur input prompt:\033[0m")
print(prompt)
print("\033[93mGenerated text:\033[0m")
if hasattr(response.choices[0].message, 'reasoning'):
    print("<think>")
    print(response.choices[0].message.reasoning)
    print("</think>")

print(response.choices[0].message.content)

You can also add all options available in the API, such as ``max_tokens`` regulating the number of tokens to predict, ``temperature`` of the model, or the ``seed`` for reproducibility.

In [ ]:
prompt = """[Verse 1]
Buddy, you're a boy, make a big noise
Playing in the street
Gonna be a big man someday
You got mud on your face
You big disgrace
Kickin' your can all over the place, singin'

[Chorus]
We will, we will rock you
We will, we will rock you

[Verse 2]
Buddy, you're a young man, hard man
Shouting in the street
Gonna take on the world someday
You got blood on your face
You big disgrace
Waving your banner all over the place

[Chorus]
We will, we will rock you (Sing it, oh)
We will, we will rock you

[Verse 3]"""

response = client.chat.completions.create(
    extra_body={},
    temperature=2,
    seed=1234567890,
    max_tokens=512,
    model="llama3.2:1b-text-q4_K_M",
    messages=[
        # this is how messages look like in the OpenAI API
        {
            "role": "user", # you always have a role: user (your prompt), assistant (the model's response), system (system prompt for instructions)
            "content": prompt
        }
    ]
)
print("\033[93mJSON response:\033[0m")
print(response.model_dump_json(indent=4))
print()
print("\033[93mOur input prompt:\033[0m")
print(prompt)
print("\033[93mGenerated text:\033[0m")
if hasattr(response.choices[0].message, 'reasoning'):
    print("<think>")
    print(response.choices[0].message.reasoning)
    print("</think>")
print(response.choices[0].message.content)

As you can see, our model outputs some kind of rubbish after increasing the temperature.

If you are unsure what temperature is, you may re-iterate last week's material.

### **Now the question is: How does this kind of model help us?**

### **Easy answer: Not at all**

We want to adapt the model to solve tasks for us, like answer questions. Therefore, we need to do define a good *prompt* for the model.

## Prompting Basics

#### Prompting VS Prompt Engineering

**Prompting** can be referred to as passing downstream tasks as textual prompts, unambiguous instructions reformulated to solve like the training data, to LLMs without further retraining while **Prompt (template) engineering** denotes the development of the most appropriate prompt to solve a task ([Kaltenpoth and Müller, 2024](https://aisel.aisnet.org/wi2024/91/); [Liu et al., 2023](https://doi.org/10.1145/3560815)).

Let's start with the process of prompt engineering.

First, let's write a function that generates a response using an ``options`` dict, and ``model_name`` that receives a prompt as input.

Second, we write another function that prints the response.

In [ ]:
# Define the model name
model_name = "llama3.2:1b-text-q4_K_M"

In [ ]:
def generate_response(prompt, model_name=model_name, options={"seed": 42, "max_tokens": 512, "temperature": 0}):    
    response = client.chat.completions.create(
        extra_body={},
        temperature=options["temperature"],
        seed=options["seed"],
        max_tokens=options["max_tokens"],
        model=model_name,
        messages=[
            # this is how messages look like in the OpenAI API
            {
                "role": "user", # you always have a role: user (your prompt), assistant (the model's response), system (system prompt for instructions)
                "content": prompt
            }
        ]
    )
    return response

def print_response(prompt, model_name=model_name, options={"seed": 0, "max_tokens": 512, "temperature": 0}):
    response = generate_response(prompt, model_name, options)
    # we are now ignoring the JSON response object and working with the reasoning and text directly
    # print("\033[93mJSON response:\033[0m")
    # print(response.model_dump_json(indent=4))
    # print()
    print("\033[93mOur input prompt:\033[0m")
    print(prompt)
    print("\033[93mGenerated text:\033[0m")
    if hasattr(response.choices[0].message, 'reasoning'):
        print("<think>")
        print(response.choices[0].message.reasoning)
        print("</think>")
    print(response.choices[0].message.content)

In [ ]:
prompt = "What were the main achievements of Albert Einstein?"
print_response(prompt)

As we can see, the answer is not really helpful. Using the recommendations of [Liu et al. (2023)](https://doi.org/10.1145/3560815) we reformulate it as written in a text describing Albert Einstein:

In [ ]:
prompt = "The main achievements of Albert Einstein were "
print_response(prompt)

This looks better. Let's try another example.

In [ ]:
prompt = "When did world war 2 start?"
print_response(prompt)

Hmm... the model completes with more questions, but we wanted an answer.

In [ ]:
prompt = "Q: When did world war 2 start?\nA: " # provide Q: A: format
print_response(prompt)

Better but not exactly the answer to the question.

Now we are going deeper into providing good *prompts*.

## From Zero-Shot Prompting to Few-Shot Prompting.

What we have done so far is called *zero-shot prompting* ([Liu et al., 2023](https://doi.org/10.1145/3560815);[Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)). Prompting for an answer directly. This mostly works in easy cases but not with more complex tasks.

Let's look at an example, where we want to get the sentiment of a very short movie "review".

In [ ]:
prompt = "'I love this movie' is a "
print_response(prompt)

Not helpful. Now lets try to give the model an example, which es referred to as *one-shot prompting* ([Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)).

In [ ]:
prompt = "'I hate this movie' is a negative comment.\n'I love this movie' is a"
print_response(prompt)

Let's try a more complex math problem.

In [ ]:
prompt = "What is 11x6/2?"
print_response(prompt)

Not helpful. Let's try it with a one-shot example.

In [ ]:
prompt = "What is 7x5? 35 \n What is 11x6/2? "
print_response(prompt)

Wrong.

Let's use *few-shot prompting*, which is simply using (one or) more examples ([Liu et al., 2023](https://doi.org/10.1145/3560815);[Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)).

Furthermore, we can combine prompting methods. In this case few-shot with QA-prompting.

In [ ]:
prompt = "Q: What is 11x2/2? A: 11 \nQ: What is 11x4/2? A: 22 \nQ: What is 11x6/2? A: "
print_response(prompt)

Wow, it worked. So let's briefly summarize what we learned:

* *Promping* refers to passing (unambiguously defined) natural language instructions to an LLM.
* *Prompt engineering* refers to defining the most appropriate prompt for a task.
* *Zero-shot prompting is just prompting the model without examples.
* *Few-Shot (One-shot) prompting* referst to prompting the model with one or more examples of the task.

As most of you will know ChatGPT, what is different in this models answer compared to those of ChatGPT?

Write it as commend below and share your answer with the seminar.

In [ ]:
# Write here your answer to the questions: What is different in this models answer compared to those of ChatGPT?
#

### Foundation Models VS Chat (Instruction) Models

![Foundation Models VS Chat Models](imgs/llms.png)

(Image adapted from ([Raschka (2024)](https://www.manning.com/books/build-a-large-language-model-from-scratch)))

As visible in the image above, a foundation model is generated by *pretraining*, as those model you trained in the last session.

After pretraining a model, you can fine-tune it to follow instructions or in a conversational manner ([Ouyang et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/b1efde53be364a73914f58805a001731-Paper-Conference.pdf)).

This is what was done with ChatGPT, Claude and other models ([OpenAI, 2022](https://openai.com/index/chatgpt/);[Ganguli et al., 2022](https://doi.org/10.48550/arXiv.2209.07858)).

## Chatting with LLMs

### Setup the chat API

Let's just load the imports and client again:

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    # here the url for the custom endpoint and the API key for authentication
	base_url=BASE_URL,  # example for ollama server
    # here the API key for authentication
	api_key=API_KEY,
)

Now let's talk to a real chat model version of Llama 3.2.

In [ ]:
# Define the model name
model_name = "llama3.2:1b"

# a small change to the function to accept messages directly
def generate_response(messages, model_name=model_name, options={"seed": 42, "max_tokens": 512, "temperature": 0}):    
    response = client.chat.completions.create(
        extra_body={},
        temperature=options["temperature"],
        seed=options["seed"],
        max_tokens=options["max_tokens"],
        model=model_name,
        messages=messages
    )
    return response

def print_response(messages, model_name=model_name, options={"seed": 0, "max_tokens": 512, "temperature": 0}):
    response = generate_response(messages, model_name, options)
    # we are now ignoring the JSON response object and working with the reasoning and text directly
    # print("\033[93mJSON response:\033[0m")
    # print(response.model_dump_json(indent=4))
    # print()
    print("\033[93mOur input prompt:\033[0m")
    print(json.dumps(messages, indent=4))
    print("\033[93mGenerated text:\033[0m")
    if hasattr(response.choices[0].message, 'reasoning'):
        print("<think>")
        print(response.choices[0].message.reasoning)
        print("</think>")
    print(response.choices[0].message.content)

In [ ]:
messages = [
    # this is how messages look like in the OpenAI API
    {
        "role": "user", # you always have a role: user (your prompt), assistant (the model's response), system (system prompt for instructions)
        "content": "What is the meaning of life?"
    }
]
print_response(messages)

The messages list needs to start with a message with the ``"system"`` role (e.g, "You are a helpful AI assistant that answers questions.") or a ``"user"`` role.

The system message or *system prompt* mostly contains general instructions that preceed task specific details ([Zhang et al., 2024](https://doi.org/10.48550/arXiv.2410.14826))

In [ ]:
messages = [
    {
        'role': 'system',
        'content': "You are an AI chatbot giving sarcastic answers.",
    },
    {
        'role': 'user',
        'content': "How are you?",
    },
]
print_response(messages)

If there is a message history, the messages should to be alternating ``"user"`` and ``"assistant"``.

In [ ]:
messages = [
    {
        'role': 'system',
        'content': "You are an AI chatbot giving sarcastic answers.",
    },
    {
        'role': 'user',
        'content': "How are you?",
    },
    {
        'role': 'assistant',
        'content': "I looooove being an AI chatbot! Always being asked the same questions over and over again. It's the best!",
    },
    {
        'role': 'user',
        'content': "What do you love the most?",
    },
]
print_response(messages=messages)

### Chain-of-Thought Prompting and Extensions

Let's think of a more complex math riddle, probably for five graders ([Williams and Huckle, 2024](https://doi.org/10.48550/arXiv.2405.19616)):

In [ ]:
my_message = "When I was 6 my sister was half my age. Now I’m 70 how old is my sister? Answer directly. Provide only the number."
messages = [
    {
        'role': 'system',
        'content': "You are a helpful and honest AI chatbot that follows user instructions and answers questions honestly and helpfully.",
    },
    {
        'role': 'user',
        'content': my_message,
    },
]
print_response(messages=messages)

As we can see, the model answers wrong, while thinking the right way.

The problem is that language models predict the next token by the probability of the previous context ([Shanahan et al., 2024](https://doi.org/10.1038/s41586-023-06647-8)).

This can lead to higher probabilities for the wrong tokens if they are learned from shortcut answers in the training data.

To prevent this, [Wei et al. (2022)](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf) introduced *Chain-of-Thought (CoT) prompting*, which asks the model to reason step-by-step before answering the question.

![Chain-of-Thought Prompting](imgs/cot.png)


As visible in this illustraion from the Paper of Wei et al. (2022), you can see that they apply few-shot prompting to incorporate the CoT reasoning into the model ([Wei et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf)).

Current models have already be trained for CoT reasoning. Therefore, we only need to instruct the model to think step-by-step.

In [ ]:
my_message = "When I was 6 my sister was half my age. Now I’m 70 how old is my sister? Think step-by-step."
messages = [
    {
        'role': 'system',
        'content': "You are a helpful and honest AI chatbot that follows user instructions and answers questions honestly and helpfully.",
    },
    {
        'role': 'user',
        'content': my_message,
    },
]
print_response(messages=messages)

Wow, the answer is still wrong but the reasoning is right. Amazing!

Sometimes CoT Prompting just leads to wrong answers, as with the example above.

Please just look at the outputs and don't remove the ``if False:``, as the code can't run on your computer.
For this example you need to install [PyTorch](https://pytorch.org/get-started/locally/), as well as transformers and bitsandbytes. (We will learn more about this libraries in the upcoming sessions.)

Then *self-consistency* is a helpful approach of prompting ([Wang et al. 2022](https://doi.org/10.48550/arXiv.2203.11171)).

Self-consistency uses CoT prompting with a more diverse decoding strategy, that replaces *greedy* decoding with a majority vote as visible in the provided in the paper of ([Wang et al. (2022)](https://doi.org/10.48550/arXiv.2203.11171)).

![Self-consistency](imgs/selfcon.png)

Let's do this with our example.

In [ ]:
if False:    
    set_seed(0)
    my_message = "When I was 6 my sister was half my age. Now I’m 70 how old is my sister? Please think step-by-step."

    messages = [
        {
            'role': 'user',
            'content': my_message,
        },
    ]

    chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    model_inputs = tokenizer(chat_prompt, return_tensors='pt').to("cuda") # this won't work on CPU

    beam_outputs = model.generate(
        **model_inputs,
        max_new_tokens=256,
        temperature=0.7, # same as ollama
        do_sample=True,
        num_return_sequences=3,
    )

In [ ]:
if False:    
    for beam_output in beam_outputs:
        print(tokenizer.decode(beam_output, skip_special_tokens=True))
        print("-" * 25)
        print("-" * 25)
print("""user
When I was 6 my sister was half my age. Now I’m 70 how old is my sister? Please think step-by-step.
* Think about how old you were when you were 6.  
* Think about how much older your sister is than you.  
* Do these things to figure out how old she is now. 


**Here's the breakdown:**

* **Step 1:** When you were 6, your sister was half your age, meaning she was 6 / 2 = 3 years old. 
* **Step 2:**  This means your sister is 3 years younger than you.
* **Step 3:** You are now 70 years old.
* **Step 4:** To find your sister's age, subtract the age difference from your current age, which is 70 - 3 = 67.

**Answer:** Your sister is 67 years old. 

-------------------------
-------------------------
user
When I was 6 my sister was half my age. Now I’m 70 how old is my sister? Please think step-by-step.
*Remember the question asks for your sister's age, not your age*

**Step 1:** When you were 6, your sister was half your age, which means she was 6 / 2 = 3 years old.

**Step 2:**  The age difference between you and your sister is 6 - 3 = 3 years.

**Step 3:** Now you are 70 years old.

**Step 4:** Since the age difference remains the same, your sister is 70 - 3 = 67 years old.


**Answer:** Your sister is 67 years old. 

-------------------------
-------------------------
user
When I was 6 my sister was half my age. Now I’m 70 how old is my sister? Please think step-by-step.
* When you were 6, your sister was half your age. 
* Therefore, your sister was 6/2 = 3 years old.
* Now you are 70 years old. 
* The age difference between you and your sister is 70-6 = 6 years.
* Therefore, your sister is 70-6 = 64 years old.



**Answer:** Your sister is 64 years old. 

-------------------------
-------------------------

""")

As we can see, there is one wrong answer, but the majority vote would be "67 years", which is right. Using this decoding strategy with a tree-like structure results in *Tree-of-Thoughts (ToT)* ([Yao et al., 2023](https://doi.org/10.48550/arXiv.2305.10601)), which is out of scope in most cases due to its heavy computation requirements.

Let's summarize:

You just learned:
* Zero-shot prompting often fails in terms of correctness ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)).
* One-shot or few-shot prompting can improve this in easy settings  ([Liu et al., 2023](https://doi.org/10.1145/3560815);[Brown et al., 2020](https://papers.nips.cc/paper_files/paper/2020/file/1457c0d6bfcb4967418bfb8ac142f64a-Paper.pdf)).
* Chain-of-Thought (CoT) prompting can improve this in more complex settings ([Wei et al., 2022](https://proceedings.neurips.cc/paper_files/paper/2022/file/9d5609613524ecf4f15af0f7b31abca4-Paper-Conference.pdf)).
* Self-consistency and Tree-of-Thoughts can further improve model responses ([Liu et al., 2022](https://doi.org/10.48550/arXiv.2110.08387);[Wang et al., 2022](https://doi.org/10.48550/arXiv.2203.11171);[Yao et al., 2023](https://doi.org/10.48550/arXiv.2305.10601)).

But most of the Chain-of-Thought improvements are given to mathematical or true-false problems.

The question is: What happens when we want to solve real(-world) tasks?

As you may remember, the topic of this session is "Solving Tasks with prompting LLMs".

And a more important question: Why is this LLM that we're using so bad?

Simple answer: It is what we call a small language model (SLM). These models can run on edge hardware and are very capable considering their small size. 

More on SLMs here ([Wang et al., 2025](https://doi.org/10.1145/3711896.3736563))

But now let's proceed to real *large* language models and more advanced prompting techniques.